Config filenames in this copy are the names on main.

# CS604 Colab launcher — Van

Runtime: **A100 40GB GPU**, High-RAM off. Images stay on Colab disk (`/content/data`). Checkpoints and logs go to the shared Drive folder **CS604_RUNS**.

The GitHub repo is private, so the clone cell will ask for a Personal Access Token (hidden). Do not paste the token into the notebook text or commit it.

**Drive:** everyone must have `CS604_RUNS` under *My Drive* (create it there, or open the [shared folder](https://drive.google.com/drive/folders/107nc4GKekmaKe38GR_8VBn3Bh_xH-ZtR) → Organize → **Add shortcut to Drive** → My Drive, name it `CS604_RUNS`). A Drive *link* is not a file path Colab can write to.

A copy of `colab_train_one-config.ipynb` for Van's runs on branch `van-dev`. Now set up for Van's AdaptFormer with trainable biases and LayerNorm weights plus LoRA rank 16 on qkv at 448 px (Task 2, `configs/adaptformer_w192_lora-r16-qkv_448px_bias-and-layernorm_cosface0.1_2.4h.yaml`, prototypes `prototypes_448.pt`; without LoRA: `METHOD = "adaptformer_w192_448px_bias-and-layernorm_fused-adamw_compile_2.4h"`). For the full fine-tune at 448 (Task 1) set `METHOD = "full_ft_448px_proto-cosine_llrd0.75_fused-adamw_compile_2.4h"` with the same `PROTO_PT`. For the full fine-tune at 384 set `METHOD = "full_ft_384px_proto-cosine_llrd0.75_fused-adamw_compile_2.4h"`, and for AdaptFormer at 384 `METHOD = "adaptformer_w192_384px_bias-and-layernorm_fused-adamw_compile_2.4h"`, both with `PROTO_PT = f"{DRIVE_OUT}/prototypes_384.pt"`; for the 336 versions use `adaptformer_w192_336px_bias-and-layernorm_fused-adamw_compile_2.4h` / `full_ft_336px_proto-cosine_llrd0.75_fused-adamw_compile_2.4h` with `PROTO_PT = f"{DRIVE_OUT}/prototypes_336.pt"`. Differences:

- `BRANCH`, `METHOD` and `TEAM` are already set.
- Prediction finds the run folder that training just created and writes `val_team_{TEAM}.txt` and `.zip` inside it (not in the shared `CS604_RUNS` root). It skips the test split until test data is released.
- With `AUTO_DISCONNECT = True`, the runtime ends after the last cell, or as soon as training fails, so an idle GPU does not keep using compute units. Drive is flushed first.

Use **Runtime → Run all**.

In [ ]:
REPO_URL = "https://github.com/keaidewoq/cs604-project.git"
REPO_DIR = "/content/repo"
DATA_DIR = "/content/data"
# Must match a folder (or shortcut) in My Drive — not the browser URL.
DRIVE_OUT = "/content/drive/MyDrive/CS604_RUNS"
BRANCH = "van-dev"
# configs/<METHOD>.yaml. AdaptFormer at 448 (Task 2); full FT at 448 (Task 1): "full_ft_448px_proto-cosine_llrd0.75_fused-adamw_compile_2.4h";
# at 384: "adaptformer_w192_384px_bias-and-layernorm_fused-adamw_compile_2.4h" / "full_ft_384px_proto-cosine_llrd0.75_fused-adamw_compile_2.4h" (PROTO_PT prototypes_384.pt).
# Previous Task 2 run without LoRA: "adaptformer_w192_448px_bias-and-layernorm_fused-adamw_compile_2.4h".
METHOD = "adaptformer_w192_lora-r16-qkv_448px_bias-and-layernorm_cosface0.1_2.4h"
TEAM = "08"
# Prototypes at the config's img_size: prototypes_448.pt / prototypes_384.pt / prototypes_336.pt.
PROTO_PT = f"{DRIVE_OUT}/prototypes_448.pt"
# Eval crop for the second prediction (with flip): 0.75 at 336, 384 and 448, 0.7 for the 224 full FT.
TTA_CROP = 0.75
# Section 4a: True runs only the ~15-minute speed check (no training), then ends the runtime.
SPEED_CHECK = False
# End the runtime after the last cell, or as soon as training fails.
AUTO_DISCONNECT = True


def end_runtime(reason):
    """Flush Drive writes, then release the GPU so it stops using compute units."""
    print(reason, flush=True)
    if not AUTO_DISCONNECT:
        return
    from google.colab import drive, runtime
    try:
        drive.flush_and_unmount()  # without this the last checkpoint can be lost
        print("Drive flushed and unmounted", flush=True)
    except Exception as error:
        print(f"Drive flush failed: {error}", flush=True)
    runtime.unassign()

## 1. Clone or pull the repo

In [ ]:
import os
from getpass import getpass

token = getpass("GitHub PAT (hidden; repo scope). Leave empty if the repo is public: ")
clone_url = REPO_URL.replace("https://", f"https://{token}@") if token else REPO_URL

if os.path.isdir(os.path.join(REPO_DIR, ".git")):
    !git -C {REPO_DIR} fetch origin
    !git -C {REPO_DIR} checkout {BRANCH}
    if token:
        !git -C {REPO_DIR} pull {clone_url} {BRANCH}
    else:
        !git -C {REPO_DIR} pull origin {BRANCH}
else:
    !git clone -b {BRANCH} {clone_url} {REPO_DIR}

%cd {REPO_DIR}
!git branch --show-current
!git rev-parse --short HEAD

## 2. Install requirements

In [ ]:
%pip install -r requirements.txt

## 3. Mount Drive and check `CS604_RUNS`

If this cell errors, add a shortcut named `CS604_RUNS` to My Drive from the [shared folder](https://drive.google.com/drive/folders/107nc4GKekmaKe38GR_8VBn3Bh_xH-ZtR).

In [ ]:
from pathlib import Path

from google.colab import drive

drive.mount("/content/drive")

out = Path(DRIVE_OUT)
if not out.is_dir():
    raise FileNotFoundError(
        f"{DRIVE_OUT} not found after mounting.\n"
        "Open https://drive.google.com/drive/folders/107nc4GKekmaKe38GR_8VBn3Bh_xH-ZtR "
        "→ Organize → Add shortcut to Drive → My Drive, and name the shortcut CS604_RUNS."
    )
out.mkdir(parents=True, exist_ok=True)
print("Writing runs to", out.resolve())
print("Existing items:", [p.name for p in out.iterdir()][:20])

## 4. Download data to `/content/data`

In [ ]:
!python scripts/download_data.py --dest {DATA_DIR}

## 4a. Speed check (optional, no training)

With `SPEED_CHECK = True` this times, for the full FT at 336 and the AdaptFormer final: the data loader alone (8 / 10 / 12 workers, colour jitter on / off), the GPU step alone on synthetic images (batch 128 x 2 and 256 x 1; plain, fused AdamW, fused AdamW + `torch.compile`), and both together. It writes `van_speed_check_<time>.json` to `CS604_RUNS`, then ends the runtime and stops *Run all* before any training. About 15 minutes; leave it `False` for a normal run.

In [ ]:
import time

if SPEED_CHECK:
    SPEED_OUT = f"{DRIVE_OUT}/van_speed_check_{time.strftime('%Y%m%d_%H%M%S')}.json"
    !python scripts/van_speed_check.py --data_root {DATA_DIR} --configs configs/full_ft_336px_proto-cosine_llrd0.75_fused-adamw_compile_2.4h.yaml configs/adaptformer_w192_336px_bias-and-layernorm_cosface0.1_2.4h.yaml --out {SPEED_OUT}
    end_runtime(f"speed check finished: {SPEED_OUT}")
    raise SystemExit("speed check only: stopping before training")
else:
    print("SPEED_CHECK = False: skipping the speed check")

## 4b. Build class prototypes (once per backbone and image size)

Van's zero-shot head (~43.9% at 224, ~42.5% at 336), built at the img_size of `configs/{METHOD}.yaml`. Run **once**, then reuse `PROTO_PT` for `prototype_probe` / `prototype_lora_r8`. Takes a while (embeds train images). Skip if the file already exists on Drive.

In [ ]:
from pathlib import Path

if Path(PROTO_PT).is_file():
    print("Already have", PROTO_PT)
else:
    !python scripts/build_prototypes.py --config configs/{METHOD}.yaml --data_root {DATA_DIR} --output {PROTO_PT}

## Review settings (table)

Loads `configs/{METHOD}.yaml` (merged with `shared_vit-b16_augreg2_defaults.yaml`) so you can check every parameter before spending a GPU hour.

In [ ]:
import pandas as pd

from src.utils import apply_overrides, load_config

methods = ["full_ft_224px_lr1e-4_batch32_2epoch", "linear_probe_224px_frozen-backbone_5epoch", "lora_r8_224px_lr1e-3_5epoch", "bitfit_224px_all-biases_5epoch", "last_2blocks_224px_whole-blocks_3epoch", "class-prototypes_augreg2_224px_cosine-head", "lora_r8_224px_proto-cosine-head", "adapter_w64_224px_5epoch", "adaptformer_w64_224px_5epoch"]
merged = {name: load_config(f"configs/{name}.yaml").to_dict() for name in methods}
keys = []
for d in merged.values():
    for k in d:
        if k not in keys:
            keys.append(k)

compare = pd.DataFrame(
    {name: [merged[name].get(k, "") for k in keys] for name in methods},
    index=keys,
)
compare.index.name = "parameter"
print("Resolved parameters per method (shared_vit-b16_augreg2_defaults.yaml already merged in):")
display(compare)

chosen = load_config(f"configs/{METHOD}.yaml")
apply_overrides(chosen, data_root=DATA_DIR, output_dir=DRIVE_OUT)
resolved = pd.DataFrame(list(chosen.to_dict().items()), columns=["parameter", "value"])
print(f"\nThis run (METHOD={METHOD!r}, plus Colab data_root / output_dir):")
display(resolved)

## 5. Train (checkpoints on Drive)

In [ ]:
import subprocess
import sys
import time

TRAIN_STARTED = time.time()
proc = subprocess.Popen(
    [sys.executable, "-u", "-m", "src.train", "--config", f"configs/{METHOD}.yaml",
     "--data_root", DATA_DIR, "--output_dir", DRIVE_OUT],
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1,
)
for line in proc.stdout:
    print(line, end="", flush=True)
if proc.wait() != 0:
    end_runtime(f"training failed (exit {proc.returncode}); see the output above")
    raise SystemExit("training failed")

## 6. Predict

Uses the newest run folder created since training started; set `RUN_NAME` by hand only to predict from an older run. Uses `last.pt`: this config trains on all images, so there is no labelled validation and no `best.pt`. Writes `val_team_{TEAM}.txt` and `.zip` into the run folder.

In [ ]:
import shutil
from pathlib import Path

RUN_NAME = None  # None: the run folder training just created
if RUN_NAME is None:
    started = globals().get("TRAIN_STARTED", 0)
    runs = [p for p in Path(DRIVE_OUT).iterdir() if p.is_dir() and p.stat().st_mtime >= started
            and (p / "config.yaml").is_file()]
    assert runs, "No run folder from this session; set RUN_NAME by hand"
    RUN_NAME = max(runs, key=lambda p: p.stat().st_mtime).name
RUN_DIR = Path(DRIVE_OUT) / RUN_NAME
CKPT = RUN_DIR / "last.pt"
assert CKPT.is_file(), f"Checkpoint missing: {CKPT}"
print("Predicting from", CKPT)

!python -m src.predict --config {RUN_DIR}/config.yaml --ckpt {CKPT} --split val --team {TEAM} --data_root {DATA_DIR} --out_dir {RUN_DIR}
shutil.make_archive(str(RUN_DIR / f"val_team_{TEAM}"), "zip", root_dir=RUN_DIR, base_dir=f"val_team_{TEAM}.txt")
print("Submit:", RUN_DIR / f"val_team_{TEAM}.zip")
if (Path(DATA_DIR) / "test.txt").is_file() and (Path(DATA_DIR) / "test").is_dir():
    !python -m src.predict --config {RUN_DIR}/config.yaml --ckpt {CKPT} --split test --team {TEAM} --data_root {DATA_DIR} --out_dir {RUN_DIR}
else:
    print("Test data not available; skipping test prediction.")

## 6b. Predict with a tighter eval crop and flip averaging (optional)

Same checkpoint; only inference changes: center crop `TTA_CROP` instead of the model's 0.9, and the softmax averaged over the image and its horizontal flip (twice the inference compute). On Van's internal holdout this added +1.5 to +1.9 points for the 3- and 6-epoch laptop runs (training's random-resized crops zoom in, so a tighter test crop matches them). AdaptFormer at 336: crop 0.75 + flip 77.13% vs 76.26% plain on the holdout. Writes a second submission to `tta_crop<TTA_CROP>_flip/` inside the run folder; the plain one above is unchanged. Set `TTA = False` to skip.

In [ ]:
TTA = True
if TTA:
    TTA_DIR = RUN_DIR / f"tta_crop{round(TTA_CROP * 100):03d}_flip"
    !python -m src.predict --config {RUN_DIR}/config.yaml --ckpt {CKPT} --split val --team {TEAM} --data_root {DATA_DIR} --out_dir {TTA_DIR} --crop_pct {TTA_CROP} --tta_flip
    shutil.make_archive(str(TTA_DIR / f"val_team_{TEAM}"), "zip", root_dir=TTA_DIR, base_dir=f"val_team_{TEAM}.txt")
    print("Submit (TTA):", TTA_DIR / f"val_team_{TEAM}.zip")
    if (Path(DATA_DIR) / "test.txt").is_file() and (Path(DATA_DIR) / "test").is_dir():
        !python -m src.predict --config {RUN_DIR}/config.yaml --ckpt {CKPT} --split test --team {TEAM} --data_root {DATA_DIR} --out_dir {TTA_DIR} --crop_pct {TTA_CROP} --tta_flip

## 6c. Predict with a zoomed view (optional)

Same checkpoint; only inference changes (`src/van_zoom.py`). After the usual pass, the region the model used for its prediction (patch tokens most similar to the predicted class) is cut from a 2x-resolution copy, enlarged and classified again; the two softmaxes are averaged. Same 2 passes per image as crop + flip. On Van's internal holdout at crop 0.75: AdaptFormer 77.95% vs 77.10% with flip; full FT at 336 79.59% vs 78.97%. Writes to `zoom_crop<TTA_CROP>/` inside the run folder. Set `ZOOM = False` to skip.

In [ ]:
ZOOM = True
if ZOOM:
    ZOOM_DIR = RUN_DIR / f"zoom_crop{round(TTA_CROP * 100):03d}"
    !python -m src.predict --config {RUN_DIR}/config.yaml --ckpt {CKPT} --split val --team {TEAM} --data_root {DATA_DIR} --out_dir {ZOOM_DIR} --crop_pct {TTA_CROP} --zoom
    shutil.make_archive(str(ZOOM_DIR / f"val_team_{TEAM}"), "zip", root_dir=ZOOM_DIR, base_dir=f"val_team_{TEAM}.txt")
    print("Submit (zoom):", ZOOM_DIR / f"val_team_{TEAM}.zip")
    if (Path(DATA_DIR) / "test.txt").is_file() and (Path(DATA_DIR) / "test").is_dir():
        !python -m src.predict --config {RUN_DIR}/config.yaml --ckpt {CKPT} --split test --team {TEAM} --data_root {DATA_DIR} --out_dir {ZOOM_DIR} --crop_pct {TTA_CROP} --zoom

## 7. Loss curves (from `colab_launcher_qq.ipynb`)

Plot the losses recorded during training, after training finishes. Save and display the plots in this experiment’s Drive folder. Validation curves require labeled validation.


In [ ]:
import subprocess

from IPython.display import display, Image

FIGURE_DIR = RUN_DIR / "figures"
subprocess.run([
    "python", "scripts/plot_curves.py",
    "--run_dir", str(RUN_DIR), "--out_dir", str(FIGURE_DIR),
], check=True)

for plot in sorted(FIGURE_DIR.glob("*.png")):
    print(plot.name)
    display(Image(filename=str(plot)))


## 8. Inference latency (from `colab_launcher_qq.ipynb`)

Load the trained checkpoint selected in section 6. Measure forward-pass latency on synthetic images at batch sizes 1 and 64, excluding preprocessing and transfers. Use 30 warm-ups and 200 timed iterations. Save results to `inference_latency.json`. This runs outside the training timer; keep GPU and measurement settings consistent between experiments.


In [ ]:
import json
import torch

from src.utils import load_config, apply_overrides, set_seed
from src.model import build_model
from src.predict import load_checkpoint
from scripts.benchmark_latency import measure

assert torch.cuda.is_available(), "A GPU runtime is required."
cfg = load_config(RUN_DIR / "config.yaml")
# Partial PEFT checkpoints require the original pretrained backbone.
checkpoint_info = torch.load(CKPT, map_location="cpu", weights_only=False)
partial = bool(checkpoint_info.get("trainable_only", False))
del checkpoint_info
apply_overrides(cfg, pretrained=partial)
set_seed(int(cfg.seed))

device = torch.device("cuda")
model = build_model(cfg).to(device)
load_checkpoint(model, CKPT, device)
model.eval()

warmup = 30
iterations = 200
use_bf16 = bool(cfg.bf16)
results = {
    "checkpoint": str(CKPT),
    "gpu": torch.cuda.get_device_name(0),
    "gpu_capacity_GiB": torch.cuda.get_device_properties(0).total_memory / 1024**3,
    "torch_version": str(torch.__version__),
    "precision": "bf16" if use_bf16 else "fp32",
    "img_size": int(cfg.img_size),
    "warmup_iterations": warmup,
    "timed_iterations": iterations,
    "scope": "Forward pass on synthetic inputs; excludes loading, preprocessing and transfers",
    "measurements": [],
}

try:
    for batch_size in (1, 64):
        batch = torch.randn(
            batch_size, 3, int(cfg.img_size), int(cfg.img_size), device=device,
        )
        with torch.autocast(device_type="cuda", dtype=torch.bfloat16, enabled=use_bf16):
            ms_per_image, _ = measure(model, batch, device, warmup, iterations)
        results["measurements"].append({
            "batch_size": batch_size,
            "ms_per_image": ms_per_image,
            "ms_per_batch": ms_per_image * batch_size,
        })
        del batch
finally:
    del model
    torch.cuda.empty_cache()

output = RUN_DIR / "inference_latency.json"
output.write_text(json.dumps(results, indent=2), encoding="utf-8")
print(json.dumps(results, indent=2))
print("Saved:", output)


## 9. End the runtime

Flushes Drive and releases the GPU when `AUTO_DISCONNECT = True`.

In [ ]:
end_runtime("all cells finished; ending the runtime")